In [14]:
import pandas as pd
import time
import asyncio
import nest_asyncio
from tenacity import retry, wait_exponential, stop_after_attempt
import aiohttp
import requests
import re


In [15]:
ObservationsData = pd.read_csv(r'../Transformed Data/ObservationsClean.csv', sep=';', index_col=0)
NativeData = pd.read_csv(r'../Transformed Data/NativeDataClean.csv', sep=';', index_col=0)

In [30]:
References = pd.DataFrame()
References['Reference'] = pd.concat([ObservationsData['Reference'], NativeData['Reference']], ignore_index=True)
References.drop_duplicates(inplace=True)
References.reset_index(drop=True, inplace=True)

# Code

In [17]:
def TitleExtract(references):
    
    def CleanRefs(reference_list):
        def Text(text):
            return re.sub(r"[^a-zA-Z0-9.,() \-]", "", text)
        return [Text(ref) for ref in reference_list]

    def Title(ref):
        TitleAfterYear = re.search(r'\(\d{4}\)\.\s*(.*?)(?:\. [A-Z]|\.$)', ref)
        if TitleAfterYear:
            return TitleAfterYear.group(1).strip()

        SentenceAfterYear = re.search(r'\(\d{4}\)\.\s*(.*)', ref)
        if SentenceAfterYear:
            return SentenceAfterYear.group(1).strip()
    
        return None
    
    references = CleanRefs(references)
    return [Title(ref) for ref in references]

In [26]:
nest_asyncio.apply()
@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))

def Extract_CrossRef(titles):
    
    async def CrossRef_Sessions(titles):
        
        async def Crossref_Search(session, title):
            
            url = "https://api.crossref.org/works"
            params = {"query.title": title, "rows": 1}
    
            def CleanText(text):
                return re.sub(r"[^a-zA-Z0-9.,() \-]", "", text)
    
            try:
                async with session.get(url, params=params) as response:
                    response.raise_for_status()
                    data = await response.json()
                    item = data['message']['items'][0]
            
                    authors = ', '.join([CleanText(f"{a.get('family', '').title()}, {
                        a.get('given', '')[:1].upper()}.").strip() for a in item.get(
                            'author', [])]) or "Unknown Author"
                    year = item.get('issued', {}).get('date-parts', [[None]])[0][0]
                    title_clean = CleanText(item.get('title', [''])[0]).title()
                    doi = item.get('DOI')
            
                    if item.get("type") == "journal-article":
                        journal = CleanText(item.get('container-title', [''])[0])
                        volume = CleanText(item.get('volume', ''))
                        pages = CleanText(item.get('page', ''))
                        
                        return f"{authors} ({year}). {title_clean}. {journal}, {volume}, {pages}. DOI: {doi}"
                        
                    if item.get("type") == "book-chapter":
                        book = CleanText(item.get('container-title', [''])[0])
                        pages = CleanText(item.get('page', ''))
                        
                        return f"{authors} ({year}). {title_clean}. In: {book}, {pages}. DOI: {doi}"
                    
                    else: 
                        journal == None, volume == None, pages == None
                        return f"{authors} ({year}). {title_clean}. DOI: {doi}"
            
                    
    
            except Exception as e:
                return None

        async with aiohttp.ClientSession() as session:
            tasks = [Crossref_Search(session, title) for title in titles]
            return await asyncio.gather(*tasks)
    return asyncio.get_event_loop().run_until_complete(CrossRef_Sessions(titles))



# Extraction

In [18]:
References['Title'] = TitleExtract(References['Reference'])

In [27]:
References['CrossRef'] = Extract_CrossRef(References['Title'])

## Error Corrections

In [ ]:
len(References[References['CrossRef'].str.contains("Unknown Author", na=False)]['Reference'])


51

In [ ]:
len(References[References['CrossRef'].isna()]['Reference'])

157

In [21]:
References.to_csv(r'../Transformed Data/ReferencesClean.csv', sep=';', index=False)